# Lab 15 - How the API Works: From Text to Response
**Section covered: 15 (How the Anthropic API Works)**

---

## What this lab covers

Section 15 of the HTML guide explains what happens inside every API call:
tokenisation -> embedding -> attention/generation -> response object.

This lab makes each stage tangible. You will:
1. See how text is broken into tokens and why that matters for cost
2. Estimate token counts before making calls
3. Inspect a real response object field-by-field
4. Understand the tool_use response structure that drives your agent loop
5. See streaming in action
6. Understand context window management and why long conversations cost more

---

## Prerequisites
- `anthropic` and `python-dotenv` installed
- `tiktoken` for tokenisation inspection: `pip install tiktoken`
  (tiktoken is OpenAI's library but uses a similar BPE algorithm to Claude)

## Step 1 - Setup

In [ ]:
import anthropic
import json
import tiktoken
from dotenv import load_dotenv

# tiktoken lets us inspect tokenisation (same BPE algorithm the model uses)
# pip install tiktoken  (if not installed)

load_dotenv()
client = anthropic.Anthropic()
print("Client and tokeniser ready")

## Step 2 - Stage 1: Tokenisation

Text is never sent as characters. It is first converted to tokens.
A token is roughly 4 characters or 0.75 words on average.
This matters because you are billed per token, not per character.

In [ ]:
# ===================================================================
# STAGE 1: TOKENISATION
# Text -> tokens -> token IDs -> what the model actually sees
# ===================================================================

# Claude uses a BPE (Byte Pair Encoding) tokeniser
# We use tiktoken (OpenAI library) to approximate it - the actual mappings
# are very similar and good enough to demonstrate the concept

try:
    enc = tiktoken.get_encoding("cl100k_base")  # GPT-4 tokeniser - close to Claude
except:
    print("tiktoken not installed. Run: pip install tiktoken")
    enc = None

def show_tokenisation(text: str):
    print(f"\nText: {repr(text)}")
    if enc:
        tokens = enc.encode(text)
        decoded = [enc.decode([t]) for t in tokens]
        print(f"Token count: {len(tokens)}")
        print(f"Token IDs:   {tokens[:20]}{'...' if len(tokens) > 20 else ''}")
        print(f"Decoded:     {decoded[:20]}{'...' if len(decoded) > 20 else ''}")
    else:
        # Rough approximation: ~4 chars per token
        approx = len(text) // 4
        print(f"Approximate token count: ~{approx}")

# See how different text types tokenise
show_tokenisation("Hello, how are you today?")
show_tokenisation("The quick brown fox jumps over the lazy dog.")
show_tokenisation("def calculate(x, y):\n    return x + y")
show_tokenisation("Kubernetes namespace configuration YAML manifest")

## Step 3 - Counting Tokens and Estimating Cost Before Calling

In [ ]:
# ===================================================================
# TOKEN COUNTING AND COST ESTIMATION
# Before calling the API - count tokens and estimate cost
# ===================================================================

def count_tokens_in_request(messages: list, tools: list = None, system: str = "") -> dict:
    """
    Estimate token count before making an API call.
    Helps you budget requests and avoid hitting context limits.
    """
    total_text = system
    for msg in messages:
        content = msg.get("content", "")
        if isinstance(content, str):
            total_text += content
        elif isinstance(content, list):
            for block in content:
                if isinstance(block, dict) and block.get("type") == "text":
                    total_text += block.get("text", "")

    if tools:
        total_text += json.dumps(tools)

    if enc:
        estimated_tokens = len(enc.encode(total_text))
    else:
        estimated_tokens = len(total_text) // 4

    # Claude Sonnet 3.x pricing (approximate, check anthropic.com for current)
    input_cost_per_1k  = 0.003   # USD per 1K input tokens
    output_cost_per_1k = 0.015   # USD per 1K output tokens
    estimated_input_cost = (estimated_tokens / 1000) * input_cost_per_1k

    return {
        "estimated_input_tokens": estimated_tokens,
        "estimated_input_cost_usd": round(estimated_input_cost, 6),
        "context_limit": 200000,
        "headroom_tokens": 200000 - estimated_tokens,
    }


# Cost estimation before calling
sample_messages = [
    {"role": "user", "content": "What is the capital of France, and what is the weather there today?"}
]
estimate = count_tokens_in_request(sample_messages, system="You are a helpful assistant.")
print("Request estimate:")
for k, v in estimate.items():
    print(f"  {k}: {v}")

print("\nTip: count tokens before sending long conversations to avoid hitting")
print("     the 200K context limit or getting an unexpected bill.")

## Step 4 - What Goes Over the Wire (Request and Response Structure)

In [ ]:
# ===================================================================
# STAGE 2 AND 3: WHAT ACTUALLY GOES OVER THE WIRE
# ===================================================================

# This is what the SDK sends to the API as JSON
sample_request = {
    "model": "claude-sonnet-4-6",
    "max_tokens": 256,
    "system": "You are a helpful assistant.",
    "messages": [
        {"role": "user", "content": "What is 2 + 2?"}
    ]
}

print("What gets sent to the API (JSON body):")
print(json.dumps(sample_request, indent=2))

print()
print("What the API returns (response structure):")
print("""
{
  "id": "msg_01XFDUDYJgAACzvnptvVoYEL",
  "type": "message",
  "role": "assistant",
  "content": [
    {
      "type": "text",
      "text": "2 + 2 = 4."
    }
  ],
  "model": "claude-sonnet-4-6",
  "stop_reason": "end_turn",
  "stop_sequence": null,
  "usage": {
    "input_tokens": 22,
    "output_tokens": 10
  }
}
""")
print("Every field you need to understand for building agents:")
print("  content      -> list of blocks (text, tool_use, image)")
print("  stop_reason  -> 'end_turn' or 'tool_use' (drives your agent loop)")
print("  usage        -> what you are billed for")

## Step 5 - A Real Live API Call: Inspect Every Field

In [ ]:
# ===================================================================
# MAKE A REAL CALL AND INSPECT EVERY FIELD
# ===================================================================

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=128,
    system="You are a helpful assistant. Answer in exactly one sentence.",
    messages=[{"role": "user", "content": "What is the speed of light?"}]
)

print("=== LIVE API RESPONSE - every field ===")
print(f"id:          {response.id}")
print(f"type:        {response.type}")
print(f"role:        {response.role}")
print(f"model:       {response.model}")
print(f"stop_reason: {response.stop_reason}")
print(f"stop_seq:    {response.stop_sequence}")
print(f"\ncontent ({len(response.content)} block(s)):")
for i, block in enumerate(response.content):
    print(f"  [{i}] type={block.type}")
    if hasattr(block, "text"):
        print(f"       text={repr(block.text)}")
    if hasattr(block, "name"):
        print(f"       name={block.name}")
    if hasattr(block, "input"):
        print(f"       input={block.input}")

print(f"\nusage:")
print(f"  input_tokens:  {response.usage.input_tokens}")
print(f"  output_tokens: {response.usage.output_tokens}")
print(f"  total_tokens:  {response.usage.input_tokens + response.usage.output_tokens}")

## Step 6 - The tool_use Response: What Your Agent Loop Reads

When `stop_reason == "tool_use"`, your agent loop must:
1. Find every `tool_use` block in the content
2. Call the tool with `block.input`
3. Return a `tool_result` with the same `block.id`

This is the complete mechanic of the agent loop.

In [ ]:
# ===================================================================
# WHAT A TOOL_USE RESPONSE LOOKS LIKE
# This is what your agent loop reads to know it needs to call a tool
# ===================================================================

tool_definition = [
    {
        "name": "get_weather",
        "description": "Get current weather for a city.",
        "input_schema": {
            "type": "object",
            "properties": {"city": {"type": "string"}},
            "required": ["city"]
        }
    }
]

response_with_tool = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=256,
    tools=tool_definition,
    messages=[{"role": "user", "content": "What is the weather in Tokyo?"}]
)

print("=== TOOL USE RESPONSE - anatomy ===")
print(f"stop_reason: {response_with_tool.stop_reason}  <- 'tool_use' triggers your agent loop")
print(f"content blocks: {len(response_with_tool.content)}")

for block in response_with_tool.content:
    print(f"\nBlock type: {block.type}")
    if block.type == "text":
        print(f"  text: {repr(block.text)}")
    if block.type == "tool_use":
        print(f"  id:    {block.id}  <- you must echo this back in tool_result")
        print(f"  name:  {block.name}  <- which tool to call")
        print(f"  input: {block.input}  <- what parameters to pass")

print("""
Your agent loop pattern:
  if stop_reason == "tool_use":
      for block in response.content:
          if block.type == "tool_use":
              result = your_tool_function(block.name, block.input)
              # Send result back with the same id
              tool_results.append({
                  "type": "tool_result",
                  "tool_use_id": block.id,   <- MUST match the id above
                  "content": result
              })
""")

## Step 7 - Streaming for Production UIs

In [ ]:
# ===================================================================
# STREAMING - for long responses in production UIs
# ===================================================================
print("STREAMING - how to show output progressively as it is generated")
print("(Instead of waiting for the full response)")
print()

with client.messages.stream(
    model="claude-sonnet-4-6",
    max_tokens=128,
    messages=[{"role": "user", "content": "Count from 1 to 10 slowly, one word per line."}]
) as stream:
    print("Streaming output (each token arrives as it is generated):")
    for text in stream.text_stream:
        print(text, end="", flush=True)

print()
print()
print("Use streaming in production UIs so users see output immediately")
print("rather than waiting 5-10 seconds for a long response to complete.")
print()
print("AWS Bedrock also supports streaming via response_stream parameter")
print("in bedrock-runtime.invoke_model_with_response_stream()")

## Step 8 - Context Window Management

In [ ]:
# ===================================================================
# CONTEXT WINDOW MANAGEMENT
# Why long conversations get expensive and hit limits
# ===================================================================

# Simulate a long conversation and show how tokens accumulate
conversation = []
token_counts = []

messages_to_simulate = [
    ("user", "Tell me about the French Revolution in one sentence."),
    ("assistant", "The French Revolution (1789-1799) was a period of radical political transformation in France that overthrew the monarchy, established a republic, and culminated in Napoleon's rise to power."),
    ("user", "What were the main causes?"),
    ("assistant", "The main causes were fiscal crisis (France was bankrupt after supporting American independence), social inequality (the Third Estate paid taxes while nobles did not), Enlightenment ideas challenging absolute monarchy, and food shortages causing widespread hunger."),
    ("user", "Who was Marie Antoinette?"),
    ("assistant", "Marie Antoinette was the Austrian-born Queen of France, wife of Louis XVI, who became a symbol of royal excess and was guillotined in 1793 during the Reign of Terror."),
]

for role, content in messages_to_simulate:
    conversation.append({"role": role, "content": content})
    estimate = count_tokens_in_request(conversation)
    token_counts.append(estimate["estimated_input_tokens"])
    print(f"After {len(conversation):2d} messages: ~{estimate['estimated_input_tokens']:,} input tokens  |  headroom: {estimate['headroom_tokens']:,}")

print()
print("Key insight: every message in the conversation is sent AGAIN with each new request.")
print("A 100-turn conversation sends ~100x more tokens than a 1-turn conversation.")
print()
print("Context management strategies:")
print("  1. Summarise old turns: replace old messages with a summary every N turns")
print("  2. Sliding window: keep only the last N messages in context")
print("  3. RAG: store conversation facts externally, retrieve only what is relevant")
print()
print("For production agents:")
print("  Track input_tokens in every response.usage")
print("  Alert when approaching 150K tokens (75% of 200K limit)")
print("  Implement summarisation before the limit is hit")

## Step 9 - The Full Picture

In [ ]:
print("""
THE FULL PICTURE: What happens in every API call
=================================================

1. TOKENISATION
   Your text is split into tokens (roughly 4 chars each).
   Words like "unbelievable" become ["un", "believ", "able"] - 3 tokens.
   This is why "write shorter prompts" saves money - fewer tokens.

2. EMBEDDING
   Each token ID is mapped to a 1024+ dimensional vector.
   Similar words have similar vectors. This is how the model "understands" meaning.
   You don't see this step - it happens inside the model.

3. ATTENTION + GENERATION
   The transformer runs attention across all tokens in context.
   It predicts one token at a time, auto-regressively.
   Temperature and sampling parameters control how deterministic it is.
   stop_reason="end_turn" = model decided to stop.
   stop_reason="tool_use" = model requesting a tool call.
   stop_reason="max_tokens" = you hit the limit you set.

4. RESPONSE OBJECT
   The generated tokens are decoded back to text.
   Wrapped in the response structure: content blocks + usage + stop_reason.
   You pay for input_tokens (everything sent) + output_tokens (everything generated).

5. FOR AGENTS - THE LOOP
   stop_reason="tool_use" -> execute tools -> send results back as "tool_result" blocks
   stop_reason="end_turn" -> return final answer to user
   This cycle repeats until end_turn or your max_turns guard fires.

BILLING FORMULA:
   Cost = (input_tokens * input_rate) + (output_tokens * output_rate)
   Claude Sonnet: approx $0.003/1K input + $0.015/1K output (check anthropic.com)
   Agents are more expensive than single calls: every tool loop re-sends the full history.

CONTEXT LIMIT:
   Claude: 200,000 tokens (200K)
   = roughly 150,000 words = 500 pages of text
   Once you hit this, older messages must be summarised or dropped.
""")

## All 15 Labs Complete

| Lab | Topic | Section |
|-----|-------|---------|
| Lab 1 | First API call | 15 |
| Lab 2 | Stateful agent with memory | 02 |
| Lab 3 | Tool chaining | 08 |
| Lab 4 | Multi-agent supervisor | 03 |
| Lab 5 | Routing agent | 08 |
| Lab 6 | FSM-controlled agent | 04 |
| Lab 7 | Evaluator-optimizer | 08 |
| Lab 8 | MCP-style tool registry | 09 |
| Lab 9 | Orchestrator fan-out | 03/08 |
| Lab 10 | Prompt chaining with gates | 08 |
| Lab 11 | Static vs agentic comparison | 05 |
| Lab 12 | RAG agent with Amazon Bedrock | 10/11 |
| Lab 13 | Prompt injection and guardrails | 12 |
| Lab 14 | Evaluation metrics harness | 13 |
| Lab 15 | API internals: tokenisation to response | 15 |

Every lab runs from Jupyter Notebook in VS Code.
Every lab uses your Anthropic API key or AWS CLI credentials.
No external services needed except Lab 12 which requires Bedrock model access.